In [ ]:
import os
import glob
import gc
import numpy as np
import tensorflow as tf
from sklearn.model_selection import GroupKFold
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from keras import layers, models, losses, regularizers
from keras.models import load_model
from keras.callbacks import ModelCheckpoint

In [ ]:
# DATA SETUP
tutti_i_file = np.array(glob.glob("dataset/data/*.npz"))
gruppi = np.array([int(os.path.basename(f).replace("window_", "").replace(".npz", "")) for f in tutti_i_file])

gkf = GroupKFold(n_splits=5)
fold_metrics = []
EPOCHS_KFOLD = 40 

print("\n" + "="*50)
print(" 🚀 INIZIO GROUP K-FOLD CROSS-VALIDATION")
print("="*50)


In [ ]:
# LOSS FUNCTIONS AND METRICS
import itertools
PERM_INDICES = tf.constant(list(itertools.permutations([0, 1, 2, 3])), dtype=tf.int32)
ROOM_DIMS = tf.constant([4.8, 7.2], dtype=tf.float32)

def hungarian_total_loss(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1) 
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)

    y_true_coords_exp = tf.expand_dims(y_true_coords, 1) 
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3]) 
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)

    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3]) 

    total_cost = coords_cost_norm + (1.5 * mask_cost_norm) 
    return tf.reduce_min(total_cost, axis=1) 

def hungarian_rmse_metres(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    
    min_coords_cost = tf.reduce_min(coords_cost, axis=1)
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    
    return tf.sqrt(min_coords_cost / num_valid_people)

def hungarian_mask_acc(y_true, y_pred):
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1)) 
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1)) 
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)
    
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)
    
    #bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3])
    
    total_cost = coords_cost_norm + (1.5 * mask_cost_norm)
    best_perm_idx = tf.argmin(total_cost, axis=1, output_type=tf.int32)
    
    batch_size = tf.shape(y_pred)[0]
    gather_nd_indices = tf.stack([tf.range(batch_size, dtype=tf.int32), best_perm_idx], axis=1)
    best_mask_pred = tf.gather_nd(y_pred_mask_perm, gather_nd_indices)
    
    return tf.reduce_mean(tf.keras.metrics.binary_accuracy(y_true_mask, best_mask_pred))

In [ ]:
# DATA ENGINE 
def load_and_process_all_files(file_list, alpha=0.02):
    X_all, Y_all = [], []
    print(f"Inizio caricamento ed EMA Decluttering di {len(file_list)} file...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   
        people_xy = data['people_xy'].astype(np.float32) # Assicuriamoci sia float
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
        mag_reshaped = mag.reshape(T, 1, 120, 18) 
        
        bg = np.copy(mag_reshaped[0])
        decluttered = np.zeros_like(mag_reshaped)
        
        for t in range(T):
            bg = alpha * mag_reshaped[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag_reshaped[t] - bg)
        
        flat_coords = people_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(decluttered)
        Y_all.append(combined_target)
        
        print(f"File {i+1}/{len(file_list)} processato.")

    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)
    return X, Y


In [ ]:
# ARCHITECTURE EEAI-NET 

def squeeze_excite_block_2d(x, filters, r=8):
    """Meccanismo di Attenzione spaziale basato su Squeeze-and-Excitation"""
    # Squeeze: estrae le statistiche globali per ogni canale
    se = layers.GlobalAveragePooling2D()(x)
    # Excitation: riduce e poi ri-espande per imparare i pesi ottimali
    se = layers.Dense(max(1, filters // r), activation='relu', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    # Reshape per applicare il broadcasting moltiplicativo
    se = layers.Reshape((1, 1, filters))(se)
    return layers.Multiply()([x, se])

def residual_reduction_module_2d_mobile(x, filters, r=8, name_prefix=""):
    # --- 1. Residual Branch (Res) ---
    # Depthwise Separable invece di Conv2D standard
    res = layers.DepthwiseConv2D(kernel_size=(1, 3), padding='same', use_bias=False, name=f"{name_prefix}_res_dw")(x)
    res = layers.BatchNormalization(name=f"{name_prefix}_res_bn1")(res)
    res = layers.ReLU(name=f"{name_prefix}_res_relu1")(res)
    res = layers.Conv2D(filters, kernel_size=(1, 1), padding='same', activation='relu', name=f"{name_prefix}_res_pw")(res)
    
    res = squeeze_excite_block_2d(res, filters, r=r)
    res = layers.Add(name=f"{name_prefix}_res_add")([res, x])

    # --- 2. Reduction Branch (Red) ---
    red1 = layers.DepthwiseConv2D(kernel_size=(1, 3), strides=(1, 2), padding='same', use_bias=False, name=f"{name_prefix}_red_dw")(res)
    red1 = layers.BatchNormalization(name=f"{name_prefix}_red_bn2")(red1)
    red1 = layers.ReLU(name=f"{name_prefix}_red_relu2")(red1)
    red1 = layers.Conv2D(filters, kernel_size=(1, 1), padding='same', activation='relu', name=f"{name_prefix}_red_pw")(red1)
    
    # red2 resta una Conv2D standard 1x1 (è già il metodo più economico)
    red2 = layers.Conv2D(filters, kernel_size=(1, 1), strides=(1, 2), padding='same', activation='relu', name=f"{name_prefix}_red_conv2")(res)
    
    out = layers.Add(name=f"{name_prefix}_red_add")([red1, red2])
    return out

def build_eeai_model_v2_rrm(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")
    
    F = 64 
    r = 8  
    
    x = layers.GaussianNoise(0.01, name="input_noise")(inputs)

    # Feature Extraction 
    x = layers.Conv2D(F, kernel_size=(1, 5), padding='same', activation='relu', name="init_conv")(x)
    
    x = residual_reduction_module_2d_mobile(x, filters=F, r=r, name_prefix="rrm1") # Bins: 120 -> 60
    x = residual_reduction_module_2d_mobile(x, filters=F, r=r, name_prefix="rrm2") # Bins: 60 -> 30
    x = residual_reduction_module_2d_mobile(x, filters=F, r=r, name_prefix="rrm3") # Bins: 30 -> 15
    x = residual_reduction_module_2d_mobile(x, filters=F, r=r, name_prefix="rrm4") # Bins: 15 -> 8
    
    x = layers.Flatten(name="flatten_features")(x)
    x = layers.Dropout(0.35, name="dropout_features")(x)
    
    common_feat = layers.Dense(128, activation='relu', name="dense_shared")(x)
    
    # 4. Multi-Head Output (Coordinate + Maschera presenze)
    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)
    
    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])
    
    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V2_RRM")

In [ ]:
# K-FOLD CICLE

for fold, (train_idx, val_idx) in enumerate(gkf.split(tutti_i_file, groups=gruppi)):
    print(f"\n--- 🔄 FOLD {fold + 1}/5 ---")
    
    train_files = tutti_i_file[train_idx]
    val_files = tutti_i_file[val_idx]
    
    X_train_raw, Y_train = load_and_process_all_files(train_files, alpha=0.02)
    X_val_raw, Y_val = load_and_process_all_files(val_files, alpha=0.02)
    
    # NORMALIZATION
    GLOBAL_MAX = np.percentile(X_train_raw, 99.5)
    X_train = np.clip(X_train_raw, 0, GLOBAL_MAX) / GLOBAL_MAX
    X_val = np.clip(X_val_raw, 0, GLOBAL_MAX) / GLOBAL_MAX
    
    del X_train_raw, X_val_raw
    gc.collect() 
    
    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, Y_train)).shuffle(5000).batch(32).prefetch(tf.data.AUTOTUNE)
    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, Y_val)).batch(32).prefetch(tf.data.AUTOTUNE)
    
    model_fold = build_eeai_model_v2_rrm() 
    model_fold.compile(
        optimizer='adam',
        loss=hungarian_total_loss, 
        metrics=[hungarian_rmse_metres, hungarian_mask_acc]
    )
    
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=0)
    early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)
    
    history = model_fold.fit(
        train_dataset, validation_data=val_dataset, epochs=EPOCHS_KFOLD,
        callbacks=[reduce_lr, early_stop], verbose=1
    )
    
    best_rmse = min(history.history['val_hungarian_rmse_metres'])
    fold_metrics.append(best_rmse)
    print(f"✅ FOLD {fold + 1} COMPLETATO. Miglior Val RMSE: {best_rmse:.4f}m")
    
    del train_dataset, val_dataset, X_train, Y_train, X_val, Y_val, model_fold
    tf.keras.backend.clear_session()
    gc.collect()

print("\n RISULTATI K-FOLD:")
print(f"RMSE Medio: {np.mean(fold_metrics):.4f}m  (Dev. Std: {np.std(fold_metrics):.4f}m)")